# Regenerate Missing `summary_*.csv` Files

Several `results_sheffield` / `results_augmented` / `results_asian*` folders
are missing their `summary_{algo}_{dataset}.csv` file — the per-muscle
`gpu_lambda_column_compare_*.ipynb` evaluation runs completed (the `df_*.csv`
files are all there) but the final "combine into summary" cell either didn't
run or didn't get downloaded.

The summary is just `mean`/`std` of dice and Hausdorff per muscle — fully
derivable from the existing `df_*.csv` files already on this machine. This
notebook reconstructs every missing summary **locally, no Lambda/GPU needed**,
and never overwrites a summary that already exists.

Muscle names are read from each `df_*.csv`'s own columns (whichever column
ends in `_dice`), not parsed from the filename — this makes it robust to any
muscle-name/algorithm-tag naming convention without needing a hardcoded
registry.

In [ ]:
import pathlib
import pandas as pd

EVAL_DIR = pathlib.Path(r'C:\Projects\dissector\eval_notebooks')

results_dirs = sorted(EVAL_DIR.glob('*/codes/results_*'))
print(f'{len(results_dirs)} results directories found')
for d in results_dirs:
    print(' ', d.relative_to(EVAL_DIR))

In [ ]:
def build_summary(csv_files):
    # Group df_*.csv files by their "{algo_tag}_{dataset}" suffix and build
    # a {muscle, n, dice_mean, dice_std, hausdorff_mean, hausdorff_std}
    # summary per group, keyed off whatever the *_dice column is named.
    groups = {}
    for csv_path in csv_files:
        df = pd.read_csv(csv_path)
        dice_cols = [c for c in df.columns if c.endswith('_dice')]
        if not dice_cols:
            print(f'  [skip] no _dice column in {csv_path.name}')
            continue
        muscle_name = dice_cols[0][:-len('_dice')]
        prefix = f'df_{muscle_name}_'
        if not csv_path.stem.startswith(prefix):
            print(f'  [skip] filename does not match expected df_{{muscle}}_... pattern: {csv_path.name}')
            continue
        algo_suffix = csv_path.stem[len(prefix):]
        groups.setdefault(algo_suffix, []).append((muscle_name, df))
    return groups


def summarize_group(muscle_dfs):
    rows = []
    for muscle_name, df in muscle_dfs:
        dice_col = f'{muscle_name}_dice'
        hd_col   = f'{muscle_name}_hausdorff'
        if dice_col not in df.columns or hd_col not in df.columns:
            print(f'  [skip] missing {dice_col}/{hd_col}')
            continue
        rows.append({
            'muscle':         muscle_name,
            'n':              len(df),
            'dice_mean':      df[dice_col].mean(),
            'dice_std':       df[dice_col].std(),
            'hausdorff_mean': df[hd_col].mean(),
            'hausdorff_std':  df[hd_col].std(),
        })
    return pd.DataFrame(rows).set_index('muscle')


print('Helpers defined.')

In [ ]:
created, skipped_existing, skipped_empty = [], [], []

for results_dir in results_dirs:
    csv_files = sorted(results_dir.glob('df_*.csv'))
    if not csv_files:
        skipped_empty.append(results_dir)
        continue

    groups = build_summary(csv_files)
    for algo_suffix, muscle_dfs in groups.items():
        summary_path = results_dir / f'summary_{algo_suffix}.csv'
        rel = summary_path.relative_to(EVAL_DIR)

        if summary_path.exists():
            skipped_existing.append(rel)
            continue

        summary = summarize_group(muscle_dfs)
        if summary.empty:
            print(f'  [skip] no usable muscle data for {rel}')
            continue

        summary.to_csv(summary_path)
        created.append(rel)
        print(f'Created -> {rel}  ({len(summary)} muscles)')

print(f'\n{len(created)} summary file(s) created.')
print(f'{len(skipped_existing)} already existed (left untouched).')
print(f'{len(skipped_empty)} results dirs had no df_*.csv files at all.')

In [ ]:
print('=== Created ===')
for p in created:
    print(' ', p)

print('\n=== Already existed (untouched) ===')
for p in skipped_existing:
    print(' ', p)

print('\n=== Empty results dirs (no df_*.csv yet — still need a Lambda run) ===')
for p in skipped_empty:
    print(' ', p.relative_to(EVAL_DIR))